In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity as c_s
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import warnings
warnings.filterwarnings('ignore')

pd.options.display.max_columns = None

In [2]:
vectorizer = TfidfVectorizer()
onehotencoder = OneHotEncoder()

In [3]:
df = pd.read_csv('final_final_res.csv')
rating_final = pd.read_csv('final_final_ratings.csv')

In [4]:

# Step 1: Check which restaurants have multiple payment methods or multiple cuisines
multiple_payment_methods = df.groupby('placeID').filter(lambda x: x['Rpayment'].nunique() > 1)
multiple_cuisines = df.groupby('placeID').filter(lambda x: x['Rcuisine'].nunique() > 1)

# Step 2: One-hot encode the 'Rpayment' and 'Rcuisine' columns
payment_dummies = pd.get_dummies(df['Rpayment'], prefix='payment')
cuisine_dummies = pd.get_dummies(df['Rcuisine'], prefix='cuisine')

# Step 3: Concatenate the one-hot encoded columns to the original DataFrame
df_encoded = pd.concat([df, payment_dummies, cuisine_dummies], axis=1)

# Step 4: Drop the original 'Rpayment' and 'Rcuisine' columns as they are now encoded
df_encoded.drop(['Rpayment', 'Rcuisine'], axis=1, inplace=True)

# Step 5: Group the DataFrame by placeID and other relevant columns, then aggregate the payment and cuisine columns
df_grouped = df_encoded.groupby(
    ['placeID', 'parking_lot', 'latitude', 'longitude', 'name',
     'address', 'city', 'state', 'country', 'price', 'Rambience', 'area'],
    as_index=False
).max()

# Step 6: Display the final DataFrame with both payment and cuisine one-hot encoded and grouped
df_grouped

# Step 7: (Optional) Check which restaurants had multiple payment methods or cuisines originally
multiple_payment_methods[['placeID', 'Rpayment']].drop_duplicates()
multiple_cuisines[['placeID', 'Rcuisine']].drop_duplicates()



,placeID,Rcuisine
8,5,Burgers
11,5,Fast_Food
21,10,Bar
24,10,Bar_Pub_Brewery
31,12,Bar
35,12,Bar_Pub_Brewery
56,20,Cafeteria
60,20,Fast_Food
64,20,Pizzeria
68,21,Bar


In [5]:
df_grouped.head()

,placeID,parking_lot,latitude,longitude,name,address,city,state,country,price,Rambience,area,Unnamed: 0.1,Unnamed: 0,payment_American_Express,payment_Carte_Blanche,payment_MasterCard-Eurocard,payment_VISA,payment_bank_debit_cards,payment_cash,cuisine_American,cuisine_Armenian,cuisine_Bakery,cuisine_Bar,cuisine_Bar_Pub_Brewery,cuisine_Breakfast-Brunch,cuisine_Burgers,cuisine_Cafe-Coffee_Shop,cuisine_Cafeteria,cuisine_Chinese,cuisine_Contemporary,cuisine_Family,cuisine_Fast_Food,cuisine_Game,cuisine_International,cuisine_Italian,cuisine_Japanese,cuisine_Mexican,cuisine_Pizzeria,cuisine_Regional,cuisine_Seafood,cuisine_Vietnamese
0,1,no,18.921785,-99.235350,Paniroles,Domingo 10 711 El Empleado,Cuernavaca,Morelos,Mexico,medium,quiet,closed,0,0,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False
1,2,no,22.149709,-100.976093,El Rincón de San Francisco,Universidad 169,San Luis Potosi,San Luis Potosi,Mexico,medium,familiar,open,3,3,False,False,True,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False
2,3,yes,23.752982,-99.168434,vips,Calle Mezquite Fracc Framboyanes,Ciudad Victoria,Tamaulipas,Mexico,medium,familiar,closed,6,6,False,False,True,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False
3,4,public,18.876011,-99.219890,Cafeteria cenidet,Interior Internado Palmira SN,Cuernavaca,Morelos,Mexico,low,quiet,closed,7,7,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
4,5,yes,22.141421,-101.013955,Mcdonalds Parque Tangamanga,Lateral Salvador Nava Martinez 3145,San Luis Potosi,San Luis Potosi,Mexico,medium,familiar,closed,13,13,False,False,True,True,False,True,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False


In [6]:
df_grouped.shape

(85, 42)

In [7]:
# Define the transformer for restaurant features
restaurant_features = ColumnTransformer(
    transformers=[
        ('cuisine', OneHotEncoder(), ['cuisine_American','cuisine_Armenian','cuisine_Bakery','cuisine_Bar','cuisine_Bar_Pub_Brewery','cuisine_Breakfast-Brunch','cuisine_Burgers','cuisine_Cafe-Coffee_Shop',
                                      'cuisine_Cafeteria','cuisine_Chinese','cuisine_Contemporary','cuisine_Family','cuisine_Fast_Food','cuisine_Game','cuisine_International',
                                      'cuisine_Italian','cuisine_Japanese','cuisine_Mexican','cuisine_Pizzeria','cuisine_Regional','cuisine_Seafood','cuisine_Vietnamese']),
        ('payment', OneHotEncoder(), ['payment_American_Express', 'payment_Carte_Blanche', 'payment_MasterCard-Eurocard', 'payment_VISA',
                                      'payment_bank_debit_cards', 'payment_cash']),
        ('parking', OneHotEncoder(), ['parking_lot']),
        ('latitude', StandardScaler(), ['latitude']),
        ('longitude', StandardScaler(), ['longitude']),
        ('price', OneHotEncoder(), ['price']),
        ('area', OneHotEncoder(), ['area']),
        ('ambience', OneHotEncoder(), ['Rambience']),
        ('name', TfidfVectorizer(), 'name'),
        ('address', TfidfVectorizer(), 'address'),
        ('city', TfidfVectorizer(), 'city'),
        ('state', TfidfVectorizer(), 'state')
    ],
    remainder='drop'
)

In [8]:
restaurant_feature_matrix = restaurant_features.fit_transform(df_grouped)
restaurant_feature_matrix

<85x395 sparse matrix of type '<class 'numpy.float64'>'
	with 3884 stored elements in Compressed Sparse Row format>

In [10]:
def create_user_profile(user_id, ratings_df, restaurants_df, feature_matrix, min_rating=1):
    # Filter ratings for the specific user and only consider ratings >= min_rating
    user_ratings = ratings_df[(ratings_df['userID'] == user_id) & (ratings_df['rating'] >= min_rating)]

    # Get the restaurant indices that the user has rated
    rated_restaurant_ids = user_ratings['placeID'].tolist()
    rated_restaurant_indices = restaurants_df[restaurants_df['placeID'].isin(rated_restaurant_ids)].index.values.tolist()

    # Select the feature vectors for these restaurants
    rated_features = feature_matrix[rated_restaurant_indices].toarray()  # Convert to dense array if needed


    # Get the corresponding ratings and reshape them
    ratings = user_ratings['rating'].values.reshape(-1, 1)

    # Weight the features by the ratings
    weighted_features = rated_features * ratings

    # Calculate the user profile by averaging the weighted features
    user_profile = np.sum(weighted_features, axis=0) / np.sum(ratings)

    return user_profile

# Example: Create a user profile for user U1001
# user_profile = create_user_profile(1, rating_final, df_grouped, restaurant_feature_matrix)
# user_profile

array([ 1.        ,  0.        ,  1.        ,  0.        ,  1.        ,
        0.        ,  1.        ,  0.        ,  1.        ,  0.        ,
        1.        ,  0.        ,  1.        ,  0.        ,  1.        ,
        0.        ,  1.        ,  0.        ,  1.        ,  0.        ,
        1.        ,  0.        ,  1.        ,  0.        ,  1.        ,
        0.        ,  1.        ,  0.        ,  1.        ,  0.        ,
        0.6       ,  0.4       ,  1.        ,  0.        ,  0.4       ,
        0.6       ,  1.        ,  0.        ,  1.        ,  0.        ,
        1.        ,  0.        ,  1.        ,  0.        ,  1.        ,
        0.        ,  1.        ,  0.        ,  0.4       ,  0.6       ,
        0.4       ,  0.6       ,  1.        ,  0.        ,  1.        ,
        0.8       ,  0.        ,  0.        ,  0.2       , -0.72637135,
        0.68410639,  0.        ,  0.        ,  1.        ,  0.6       ,
        0.4       ,  0.6       ,  0.4       ,  0.        ,  0.  

In [ ]:
def create_user_profile2(user_id, ratings_df, restaurants_df, feature_matrix, userProfileDetails, min_rating=1):
    # Filter ratings for the specific user and only consider ratings >= min_rating
    user_ratings = ratings_df[(ratings_df['userID'] == user_id) & (ratings_df['rating'] >= min_rating)]
    
    # Get the restaurant indices that the user has rated
    rated_restaurant_ids = user_ratings['placeID'].tolist()
    rated_restaurant_indices = restaurants_df[restaurants_df['placeID'].isin(rated_restaurant_ids)].index.values.tolist()
    
    # Select the feature vectors for these restaurants
    rated_features = feature_matrix[rated_restaurant_indices].toarray()  # Convert to dense array if needed
    
    # Get the corresponding ratings and reshape them
    ratings = user_ratings['rating'].values.reshape(-1, 1)

    # Weight the features by the ratings
    weighted_features = rated_features * ratings
    
    # Calculate the user's profile based on restaurant features
    user_profile_from_ratings = np.sum(weighted_features, axis=0) / np.sum(ratings)

    # Now consider Rcuisine from userProfileDetails (if it exists)
    if user_id in userProfileDetails['userID'].values:
        # Get the Rcuisine of the user from userProfileDetails
        user_rcuisine = userProfileDetails[userProfileDetails['userID'] == user_id]['Rcuisine'].values[0]
        
        # Use the one-hot encoded Rcuisine columns from restaurants_df (consistent with pd.get_dummies)
        cuisine_columns = [col for col in restaurants_df.columns if col.startswith('cuisine_')]
        
        # Create a one-hot encoded vector for the user's Rcuisine using pd.get_dummies format
        user_rcuisine_one_hot = np.zeros(len(cuisine_columns))
        
        # Check if the user's Rcuisine matches any of the one-hot encoded columns
        matching_column = f'cuisine_{user_rcuisine}'
        if matching_column in cuisine_columns:
            idx = cuisine_columns.index(matching_column)
            user_rcuisine_one_hot[idx] = 1  # Set the matching cuisine to 1
        
        # You can give the user's Rcuisine a certain weight (e.g., 1) to combine it with the weighted features
        user_rcuisine_weight = 1
        
        # Add the one-hot encoded user's Rcuisine to the weighted sum of features
        user_profile = (user_profile_from_ratings + user_rcuisine_one_hot * user_rcuisine_weight) / (1 + user_rcuisine_weight)
    
    else:
        # If no Rcuisine is available, return the profile based on ratings alone
        user_profile = user_profile_from_ratings
    
    return user_profile


In [11]:
user_profiles = {}

for user_id in rating_final['userID'].unique():
    user_profile = create_user_profile(user_id, rating_final, df_grouped, restaurant_feature_matrix)

    # Store the user profile in the dictionary
    user_profiles[user_id] = user_profile

user_profiles_df = pd.DataFrame(user_profiles).T

user_profiles_df.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,263,264,265,266,267,268,269,270,271,272,273,274,275,276,277,278,279,280,281,282,283,284,285,286,287,288,289,290,291,292,293,294,295,296,297,298,299,300,301,302,303,304,305,306,307,308,309,310,311,312,313,314,315,316,317,318,319,320,321,322,323,324,325,326,327,328,329,330,331,332,333,334,335,336,337,338,339,340,341,342,343,344,345,346,347,348,349,350,351,352,353,354,355,356,357,358,359,360,361,362,363,364,365,366,367,368,369,370,371,372,373,374,375,376,377,378,379,380,381,382,383,384,385,386,387,388,389,390,391,392,393,394
1,1.0,0.0,1.0,0.0,1.0,0.0,1.000000,0.000000,1.000000,0.000000,1.0,0.0,1.0,0.0,1.0,0.0,1.000000,0.000000,1.000000,0.000000,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.6,0.4,1.000000,0.000000,0.4,0.6,1.0,0.0,1.0,0.0,1.000000,0.000000,1.0,0.0,1.000000,0.000000,1.0,0.0,0.400000,0.600000,0.400000,0.600000,1.0,0.0,1.0,0.800000,0.000000,0.0,0.200000,-0.726371,0.684106,0.000000,0.000000,1.000000,0.6,0.4,0.600000,0.400000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.129679,0.0,0.0,0.0,0.0,0.0,0.0,0.120735,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.200635,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.00000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.4,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.219312,0.0,0.0,0.0,0.200635,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.178885,0.0,0.0,0.282843,0.0,0.000000,0.0,0.0,0.0,0.00000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.178885,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.097105,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.178885,0.0,0.178885,0.0,0.0,0.178885,0.0,0.0,0.0,0.0,0.0,0.100947,0.100947,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.100947,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.000000,0.0,0.000000,0.0,0.0,0.282843,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.143246,0.400000,0.0,0.230940,0.230940,0.230940,0.0,0.139572,0.230940,0.0,0.400000,0.230940,0.230940,0.2
2,1.0,0.0,1.0,0.0,1.0,0.0,0.666667,0.333333,1.000000,0.000000,1.0,0.0,1.0,0.0,1.0,0.0,0.666667,0.333333,1.000000,0.000000,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.000000,0.000000,1.0,0.0,1.0,0.0,1.0,0.0,0.666667,0.333333,1.0,0.0,0.666667,0.333333,1.0,0.0,0.666667,0.333333,0.666667,0.333333,1.0,0.0,1.0,0.000000,0.333333,0.0,0.666667,-0.826612,0.102832,0.000000,0.333333,0.666667,1.0,0.0,0.666667,0.333333,0.0,0.0,0.166991,0.144019,0.0,0.0,0.0,0.

In [12]:
from sklearn.metrics.pairwise import cosine_similarity

def calculate_similarity(user_vector, restaurant_features):
    return cosine_similarity(user_vector, restaurant_features)


In [13]:
def recommend(userid ,userProfile ,restaurant_feature_matrix ,top_n = 20):
    user = userProfile.loc[userid].values.reshape(1,-1)
    similarity = calculate_similarity(user ,restaurant_feature_matrix)
    top = similarity.argsort()[0][-top_n:][::-1]
    return df.iloc[top]
    # recommended_restaurants = df.iloc[top]

    # Drop duplicate restaurants based on 'placeID'
    # unique_recommendations = recommended_restaurants.drop_duplicates(subset='placeID')

    # return unique_recommendations

In [14]:
class RecommendationModel:
    def __init__(self, user_profiles, restaurant_features):
        self.user_profiles = user_profiles
        self.restaurant_features = restaurant_features

    def calculate_similarity(self, user_vector):
        return cosine_similarity(user_vector, self.restaurant_features)

    def recommend(self, user_id, top_n=30):
        user_vector = self.user_profiles.loc[user_id].values.reshape(1, -1)
        similarity = self.calculate_similarity(user_vector)

        top_indices = similarity.argsort()[0][-top_n:][::-1]
        # return df.iloc[top_indices]
        recommended_restaurants = df.iloc[top_indices]

        unique_recommendations = recommended_restaurants.drop_duplicates(subset='placeID')

        return unique_recommendations['placeID'].tolist()

In [21]:
rec_model = RecommendationModel(user_profiles_df, restaurant_feature_matrix)

# recommended_restaurants = recommendation_model.recommend(2)
# recommended_restaurants

In [22]:
import pickle

# Save the model
with open('rec_model.pkl', 'wb') as f:
    pickle.dump(rec_model, f)

print("Model saved to trained_model.pkl")

Model saved to trained_model.pkl


In [189]:
# recommendation_model = RecommendationModel(user_profiles_df, restaurant_feature_matrix)

# recommended_restaurants = recommendation_model.recommend('U1067', top_n=5).drop_duplicates(subset='placeID')
# recommended_restaurants


In [190]:
recommend('U1067' ,user_profiles_df ,restaurant_feature_matrix,5)

,placeID,Rcuisine,Rpayment,parking_lot,latitude,longitude,name,address,city,state,country,price,Rambience,area
0,135109,Italian,cash,no,18.921785,-99.235350,Paniroles,Domingo 10 711 El Empleado,Cuernavaca,Morelos,Mexico,medium,quiet,closed
3,135106,Mexican,MasterCard-Eurocard,no,22.149709,-100.976093,El Rincón de San Francisco,Universidad 169,San Luis Potosi,San Luis Potosi,Mexico,medium,familiar,open
2,135106,Mexican,VISA,no,22.149709,-100.976093,El Rincón de San Francisco,Universidad 169,San Luis Potosi,San Luis Potosi,Mexico,medium,familiar,open
15,135079,Chinese,cash,no,22.156376,-100.998355,Koye Sushi,Nereo Rodriguez Barragan 450 E Centro,San Luis Potosi,San Luis Potosi,Mexico,high,familiar,closed
13,135086,Fast_Food,MasterCard-Eurocard,yes,22.141421,-101.013955,Mcdonalds Parque Tangamanga,Lateral Salvador Nava Martinez 3145,San Luis Potosi,San Luis Potosi,Mexico,medium,familiar,closed


In [191]:
recommendations = recommend('U1067', user_profiles_df, restaurant_feature_matrix)

# Ensure the results are unique based on 'placeID'
unique_recommendations = recommendations.drop_duplicates(subset='placeID')

# Display the unique recommendations
unique_recommendations

,placeID,Rcuisine,Rpayment,parking_lot,latitude,longitude,name,address,city,state,country,price,Rambience,area
0,135109,Italian,cash,no,18.921785,-99.235350,Paniroles,Domingo 10 711 El Empleado,Cuernavaca,Morelos,Mexico,medium,quiet,closed
3,135106,Mexican,MasterCard-Eurocard,no,22.149709,-100.976093,El Rincón de San Francisco,Universidad 169,San Luis Potosi,San Luis Potosi,Mexico,medium,familiar,open
15,135079,Chinese,cash,no,22.156376,-100.998355,Koye Sushi,Nereo Rodriguez Barragan 450 E Centro,San Luis Potosi,San Luis Potosi,Mexico,high,familiar,closed
13,135086,Fast_Food,MasterCard-Eurocard,yes,22.141421,-101.013955,Mcdonalds Parque Tangamanga,Lateral Salvador Nava Martinez 3145,San Luis Potosi,San Luis Potosi,Mexico,medium,familiar,closed
6,135104,Mexican,MasterCard-Eurocard,yes,23.752982,-99.168434,vips,Calle Mezquite Fracc Framboyanes,Ciudad Victoria,Tamaulipas,Mexico,medium,familiar,closed
82,135044,Chinese,cash,no,22.141848,-100.997475,Restaurant Wu Zhuo Yi,Himno Nacional 100 Avenida,San Luis Potosi,San Luis Potosi,Mexico,medium,familiar,closed
14,135085,Fast_Food,cash,public,22.150802,-100.982680,Tortas Locas Hipocampo,Venustiano Carranza 719 Centro,San Luis Potosi,San Luis Potosi,Mexico,medium,familiar,closed
7,135088,Cafeteria,cash,public,18.876011,-99.219890,Cafeteria cenidet,Interior Internado Palmira SN,Cuernavaca,Morelos,Mexico,low,quiet,closed
22,135073,Bar,VISA,yes,22.147175,-100.974269,Restaurante Bar El Gallinero,Pascual M. Hernandez 210 Centro,San Luis Potosi,San Luis Potosi,Mexico,high,familiar,closed
40,135060,Seafood,cash,no,22.156883,-100.978485,Restaurante Marisco Sam,Ignacio Allende 785 Centro,San Luis Potosi,San Luis Potosi,Mexico,medium,familiar,closed


In [192]:
recommend('U1106' ,user_profiles_df ,restaurant_feature_matrix ,8)

,placeID,Rcuisine,Rpayment,parking_lot,latitude,longitude,name,address,city,state,country,price,Rambience,area
58,135053,Cafeteria,MasterCard-Eurocard,yes,22.178931,-101.012861,La Fontana Pizza Restaurante and Cafe,Satelite 606 Satelite,San Luis Potosi,San Luis Potosi,Mexico,high,familiar,closed
16,135079,Chinese,American_Express,no,22.156376,-100.998355,Koye Sushi,Nereo Rodriguez Barragan 450 E Centro,San Luis Potosi,San Luis Potosi,Mexico,high,familiar,closed
24,135073,Bar_Pub_Brewery,cash,yes,22.147175,-100.974269,Restaurante Bar El Gallinero,Pascual M. Hernandez 210 Centro,San Luis Potosi,San Luis Potosi,Mexico,high,familiar,closed
40,135060,Seafood,cash,no,22.156883,-100.978485,Restaurante Marisco Sam,Ignacio Allende 785 Centro,San Luis Potosi,San Luis Potosi,Mexico,medium,familiar,closed
39,135069,Bar,cash,yes,22.140129,-100.944872,Abondance Restaurante Bar,Industrias 908 Valle Dorado,San Luis Potosi,San Luis Potosi,Mexico,low,familiar,closed
60,135053,Fast_Food,cash,yes,22.178931,-101.012861,La Fontana Pizza Restaurante and Cafe,Satelite 606 Satelite,San Luis Potosi,San Luis Potosi,Mexico,high,familiar,closed
63,135053,Fast_Food,American_Express,yes,22.178931,-101.012861,La Fontana Pizza Restaurante and Cafe,Satelite 606 Satelite,San Luis Potosi,San Luis Potosi,Mexico,high,familiar,closed
71,135052,Bar_Pub_Brewery,cash,no,22.150981,-100.977412,La Cantina Restaurante,Ignacio Aldama 300 Centro,San Luis Potosi,San Luis Potosi,Mexico,high,familiar,closed


In [20]:
df[df['placeID'].isin(recommended_restaurants)]['name'].unique()

array(['vips', 'Cafeteria cenidet', 'Mcdonalds Parque Tangamanga',
       'Tortas Locas Hipocampo', 'Koye Sushi',
       'Restaurante la Parroquia Potosina',
       'Restaurante Bar El Gallinero', 'Sushi Itto',
       'Restaurante Tiberius', 'El Herradero Restaurante and Bar',
       'la Cochinita Pibil Restaurante Yucateco',
       'Restaurante y Pescaderia Tampico',
       'La Fontana Pizza Restaurante and Cafe', 'La Cantina Restaurante',
       'pizza clasica'], dtype=object)